In [1]:
%%capture

%pip install -q \
    "ragas>=0.2.15" \
    "langchain-community<0.4.2" \
    langchain_core \
    langchain_huggingface \
    datasets pandas matplotlib seaborn

In [2]:
import os
import asyncio
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragas.metrics.collections import SummaryScore, SemanticSimilarity, AnswerCorrectness

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [18]:
MODEL_ID = "Qwen/Qwen3-4B" # "Qwen/Qwen3-14B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,      # low temperature for deterministic judging
    do_sample=False,
    repetition_penalty=1.05,
)

hf_llm = HuggingFacePipeline(pipeline=pipe)
judge_llm = LangchainLLMWrapper(hf_llm)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/tmp/ipykernel_1605/712302237.py:22: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(hf_llm)


In [ ]:
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-8B"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_ID,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
judge_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model loaded: Qwen/Qwen3-Embedding-8B


/tmp/ipykernel_1605/3010411902.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


In [ ]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train").select(range(1))

df = ds.to_pandas()

README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 93.5kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22 [00:00<?, ? examples/s]

In [9]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [28]:
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.run_config import RunConfig

metrics = [
    SummarizationScore(llm=judge_llm, coeff=0.5),  # 0.5 balances QA vs. conciseness
    SemanticSimilarity(embeddings=judge_embeddings),
    AnswerCorrectness(llm=judge_llm, embeddings=judge_embeddings)
]

run_config = RunConfig(timeout=900, max_workers=1, max_retries=3)

base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
)

/tmp/ipykernel_1605/4277389163.py:1: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1605/4277389163.py:1: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1605/4277389163.py:1: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summarizat

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [ ]:
# TODO: remove cot

In [26]:
model.device

device(type='cuda', index=0)

In [27]:
hf_embeddings

HuggingFaceEmbeddings(model_name='Qwen/Qwen3-Embedding-8B', cache_folder=None, model_kwargs={'device': 'cuda'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [21]:
import time

t = time.perf_counter()
out = pipe("What is the capital of France? Answer briefly.")
print(f"short gen: {time.perf_counter() - t:.1f}s")

long_text = "context " * 2000   # имитация длинного RAGAS-промпта
t = time.perf_counter()
out = pipe(long_text)
print(f"long gen: {time.perf_counter() - t:.1f}s")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


short gen: 27.9s
long gen: 28.0s


In [22]:
import time
t = time.perf_counter()
hf_embeddings.embed_documents(["some text " * 500] * 4)
print(f"embed batch: {time.perf_counter() - t:.1f}s")

embed batch: 0.4s


In [23]:
import ragas
print(ragas.__version__)

0.4.3


In [ ]:
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))

### prev

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from ragas import evaluate, EvaluationDataset 
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.dataset_schema import SingleTurnSample
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_cohere import ChatCohere
from ragas.llms import LangchainLLMWrapper
from getpass import getpass

/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summari

In [5]:
COHERE_API_KEY = getpass("Enter your Cohere API key: ")

In [4]:
GOOGLE_API_KEY = getpass("Enter your GOOGLE API key: ")

In [ ]:
# cohere_llm = ChatCohere(
#     cohere_api_key=COHERE_API_KEY,
#     model="command-a-03-2025",
#     temperature=0,
#     max_tokens=4096,          # RAGAS rubrics can be verbose
# )

In [6]:
import cohere

cohere_client = cohere.ClientV2(COHERE_API_KEY)

In [7]:
from ragas.llms import llm_factory

judge_llm = llm_factory(
    "command-a-03-2025",           # model name
    provider="cohere",             # tell RAGAS this is Cohere
    client=cohere_client,          # pass the raw Cohere client
    adapter="litellm",             # <-- the key: force LiteLLM adapter
    temperature=0,
    max_tokens=8192,
)

In [8]:
# embeddings = LangchainEmbeddingsWrapper(
#     GoogleGenerativeAIEmbeddings(
#         model="gemini-embedding-001",  # or "models/embedding-001"
#         google_api_key=GOOGLE_API_KEY,
#     )
# )

from google import genai
from ragas.embeddings import GoogleEmbeddings

client = genai.Client(api_key=GOOGLE_API_KEY)

# 2a. Option A — GoogleEmbeddings directly (explicit client)
gemini_embeddings = GoogleEmbeddings(
    client=client,
    model="gemini-embedding-001",   # or "text-embedding-004"
)

In [9]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train").select(range(1))
df = ds.to_pandas()

# Show the columns and a sample row
print(df.columns.tolist())
df.head(2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


['user_input', 'reference', 'base_response', 'ft_response']


,user_input,reference,base_response,ft_response
0,Original Post: Help with Small living room Use...,The OP asked about general design suggestions ...,"The original poster, moving into a 1920s craft...",The user is asking for advice on how to arrang...


In [11]:
def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [12]:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision

# Instantiate each metric with the judge LLM
faithfulness_metric = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=gemini_embeddings)
context_precision_metric = ContextPrecision(llm=judge_llm)

In [16]:
async def score_sample(sample):
    return {
        "faithfulness": (await faithfulness_metric.ascore(
            user_input=sample["question"],
            response=sample["answer"],
            retrieved_contexts=sample["contexts"],
        )).value,
        "answer_relevancy": (await answer_relevancy_metric.ascore(
            user_input=sample["question"],
            response=sample["answer"],
        )).value,
        "context_precision": (await context_precision_metric.ascore(
            user_input=sample["question"],
            retrieved_contexts=sample["contexts"],
            reference=sample["ground_truth"],
        )).value,
    }

In [17]:
import asyncio

async def run_evaluation(dataset):
    results = []
    for sample in dataset:
        # Add a small delay to respect rate limits
        await asyncio.sleep(1) 
        try:
            scores = await score_sample(sample)
            results.append({**sample, **scores})
        except Exception as e:
            print(f"Error scoring sample: {e}")
            results.append({**sample, "error": str(e)})
    return results

In [18]:
scored_results = await run_evaluation(ds)

Error scoring sample: 'question'


In [15]:
metrics = [
    faithfulness_metric,
    answer_relevancy_metric,
    context_precision_metric,
]

base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=gemini_embeddings,
)

TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]

In [ ]:
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))